In [ ]:
import torch
from torch import Tensor
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset

def byte_pair_encoding(text, vocabulary_size):
    vocabulary = {}
    most_frequent = None
    replacement = 888 # First unused unicode codepoint
    tokens = text.encode("utf-8")

    # Tokenize text into a list of iteratively compressed bytes
    while len(vocabulary) < vocabulary_size:
        for i in range(0, len(tokens) - 1):
            group = (tokens[i], tokens[i + 1])
            if group not in vocabulary:
                vocabulary[group] = [i]
            else:
                vocabulary[group].append(i)

            if most_frequent is None or len(vocabulary[group]) > len(vocabulary[most_frequent]):
                most_frequent = group

        indexes = vocabulary[most_frequent]
        for i in indexes:
            tokens = [*tokens[:i], replacement, *tokens[i + 2:]]
        replacement += 1

    return tokens, vocabulary

class WMT(Dataset):
    def __init__(self, data_split, num_rows, vocabulary_size):
        super().__init__()
        dataset = load_dataset("wmt/wmt14", "fr-en", split=f"{data_split}[:{num_rows}]")
        self.df = dataset.data.to_pandas()
        self.rows: list[tuple[Tensor, Tensor, dict, dict] | None] = [None for _ in range(num_rows)]
        self.vocab_size = vocabulary_size

    def __len__(self):
        return self.df.shape[0]

    def __getitem__(self, idx):
        if self.rows[idx] is None:
            c = self.df.columns[0]
            en, fr = self.df[c].str["en"][idx], self.df[c].str["fr"][idx]
            en_tokens, en_vocabulary = byte_pair_encoding(en, self.vocab_size)
            fr_tokens, fr_vocabulary = byte_pair_encoding(fr, self.vocab_size)
            self.rows[idx] = (
                torch.tensor(en_tokens), torch.tensor(fr_tokens),
                en_vocabulary, fr_vocabulary)
        return self.rows[idx][0], self.rows[idx][1]

In [ ]:
train_samples = WMT("train", 64, 10)
train_loader = DataLoader(dataset=train_samples, batch_size=1, shuffle=True)
for en_token, fr_token in train_loader:
    print(en_token, fr_token)